In [ ]:
%matplotlib tk

import pandas as pd
import numpy as np
import matplotlib.pylab as plt

# Jupyter loads files from the directory the script is located in.
base_filename = 'initial_walk_test_10-08-2026_16-31-45_3.csv'
save_file = f'../data/zero_velocity_windows/{base_filename}'
df_long = pd.read_csv(f'../data/measured_walks/{base_filename}')
ground_truth = pd.read_csv('../data/ZVW_hand_labels/ground_truth_ZVW_long_walk.csv')

In [2]:
def detect_zvw(df, acc_thresh, gyro_thresh, var_thresh, var_window, min_dwell):
    acc_mag = np.sqrt(df['ax']**2 + df['ay']**2 + df['az']**2)
    gyro_mag = np.sqrt(df['gx']**2 + df['gy']**2 + df['gz']**2)

    # near the threshold?
    acc_thresh_samples = np.abs(acc_mag - 1) <= acc_thresh
    # rotating?
    gyro_mag_samples = gyro_mag <= gyro_thresh

    # Rolling variance (.rolling <- creates a rolling window + .var() <- calculates the variance)
    # is it vibrating or sliding?
    # .fillna(1.0) - first samples of data dont have enough prev points to fill a window
    # so this add 1.0 values instead of the empty NaN values to fill the window.
    acc_var = acc_mag.rolling(window=var_window).var().fillna(1.0)
    var_cond = acc_var <= var_thresh

    is_zvw = acc_thresh_samples & gyro_mag_samples & var_cond

    # Create a unique ID for every consecutive block of True's or False's
    blocks = (is_zvw != is_zvw.shift()).cumsum()

    # Group by that ID, sum the True's, and check if it meets min_dwell
    final_zvw = is_zvw.groupby(blocks).transform('sum') >= min_dwell

    # True and the previous is False
    is_start = final_zvw & ~final_zvw.shift(1, fill_value=False)
    # True and the following if False
    is_end = final_zvw & ~final_zvw.shift(-1, fill_value=False)

    zvw_start_idxs = df['seq'][is_start].tolist()
    zvs_end_idxs = df['seq'][is_end].tolist()

    return acc_mag, gyro_mag, (zvw_start_idxs, zvs_end_idxs)

In [39]:
ACC_DEVIATION = 0.3    # Allowable deviation from 1.0g
GYRO_LIMIT = 75.0       # Maximum dps
VAR_LIMIT = 0.04        # Maximum rolling variance
VAR_WINDOW = 15         # Variance window to calculate
DWELL = 5               # Minimum consecutive samples

acc_mag, gyro_mag, predicted_zvws = detect_zvw(df_long, ACC_DEVIATION, GYRO_LIMIT, VAR_LIMIT, VAR_WINDOW, DWELL)

In [41]:
fig, ax1 = plt.subplots(figsize=(16,6))
ax1.plot(df_long['seq'], acc_mag, color="tab:blue", label='Acc Mag')
ax1.set_xlabel('Sequence')
ax1.set_ylabel('Acceleration Magnitude')

ax2 = ax1.twinx()
ax2.plot(df_long['seq'], gyro_mag, color='tab:red', label='Gyro Mag')
ax2.set_xlabel('Sequence')
ax2.set_ylabel('Gyroscope Magnitude')

starts, ends = predicted_zvws

for i, (start, end) in enumerate(zip(starts, ends)):
    if i == 0:
     ax1.axvspan(start, end, color='y', alpha=0.3, label='Predicted ZVWs')
    else:
       ax1.axvspan(start, end, color='y', alpha=0.3)
    

lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper right')

plt.title('Zero-Velocity Windows')
plt.tight_layout()

plt.savefig(f"{save_file.replace('.csv', '_ZVW.png')}", dpi=120)

plt.show()